# 01 — Ingest: 51 GiB of CSV to clean Parquet

**What this notebook does:** reads all 28 raw CSVs once, fixes the three quirks
that would otherwise corrupt the data, nulls physically impossible readings, and
writes one Parquet file per dataset-year to `data/interim/`.

**Why bother.** Two reasons, and the second matters more than the first.

*Speed.* Everything after this reads Parquet instead of CSV. The whole archive
shrinks from 51 GiB to roughly 1 GiB and a query that took a minute takes a
second — which is the difference between running an experiment and deciding not
to bother.

*One definition of "clean".* If each notebook parses the CSVs itself, each one
gets to decide what `NULL` means and whether 762 mm of rain in a day is real.
Doing it once, here, means every downstream number rests on the same decisions,
and those decisions live in `config/config.yaml` where you can see them.

**What this notebook deliberately does NOT do**

- It does not drop excluded stations. `FW.PKG.01` is excluded from *canal
  averages* because it is a Chao Phraya river gauge, not because it is bad data.
  Exclusions belong in feature building (notebook 05), not in ingestion. Throwing
  data away at the front door is irreversible.
- It does not resample, fill gaps or impute. Missing stays missing. `NaN` is
  information — it means "the sensor was offline" — and a model that cannot tell
  that from "the road was dry" will learn the wrong lesson.

**Runtime:** roughly 3 minutes. Memory never exceeds the 2 GB cap in
`config.yaml`; DuckDB streams and spills rather than loading a year at a time.

## Setup

In [1]:
import os, sys, time, json
from pathlib import Path

_here = Path.cwd()
_root = next(p for p in [_here, *_here.parents] if (p / "config/config.yaml").is_file())
sys.path.insert(0, str(_root / "src"))
os.chdir(_root)

import pandas as pd
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

from bkkflood.config import load_config
from bkkflood.rawio import (DATASETS, connect, raw_file, ingest_year_to_parquet,
                            interim_path, interim_sql, verify_ordering)

CFG = load_config()
REPORTS = Path(CFG["paths"]["reports"]) / "phase0"
REPORTS.mkdir(parents=True, exist_ok=True)
INTERIM = Path(CFG["paths"]["interim"])
INTERIM.mkdir(parents=True, exist_ok=True)

print("memory cap    :", CFG["compute"]["memory_limit"])
print("threads       :", CFG["compute"]["threads"])
print("compression   :", CFG["compute"]["parquet_compression"])
print("output folder :", INTERIM.resolve())

memory cap    : 2GB
threads       : 4
compression   : zstd
output folder : /sessions/compassionate-busy-dijkstra/mnt/bkk-flood-forecast/data/interim


## 1. What "clean" means here, precisely

Five things happen to every file, and nothing else.

| Step | What it fixes |
|---|---|
| Read with `utf-8-sig` | strips the byte-order mark, so the first column is `rain_code` and not `\ufeffrain_code` |
| `nullstr='NULL'` | the literal text `NULL` becomes a real missing value instead of poisoning the column into text |
| Quoted CSV parsing | Thai station names contain commas; without quoting, every field after the name shifts |
| Explicit `DOUBLE` types | a column that is 99.6% missing (`wl_out02`) would otherwise be sniffed as text |
| Range checks | physically impossible readings become missing, **and are counted** |

### On range checks, and why we null rather than clip

`RF.BKY.02` reports 762 mm of rain in 24 hours in 2023. That same gauge's largest
hourly reading all year is 73 mm. 762 mm is not a big storm; it is a broken
record. Clipping it to the 400 mm ceiling would invent a rainfall event that never
happened and hand it to the model as truth. Nulling it says the honest thing: we
do not know what fell here.

The thresholds are in `config.yaml` under `exclusions.range_checks`, and every
value they remove is counted and reported below rather than disappearing quietly.

In [2]:
rc = CFG["exclusions"]["range_checks"]
pd.DataFrame([
    {"dataset": ds, "column": col, "min": r["min"], "max": r["max"]}
    for ds, cols in rc.items() for col, r in cols.items()
])

,dataset,column,min,max
0,rain,rf5min,0.0,60.0
1,rain,rf15min,0.0,60.0
2,rain,rf30min,0.0,100.0
3,rain,rf1hr,0.0,150.0
4,rain,rf3hr,0.0,250.0
5,rain,rf6hr,0.0,300.0
6,rain,rf12hr,0.0,350.0
7,rain,rf24hr,0.0,400.0
8,flood,flood,0.0,200.0
9,water,wl_in,-10.0,10.0


## 2. Ingest

One pass per file. The output keeps a common shape across all four datasets —
`station_code`, `station_name`, `ts`, then the value columns — so downstream code
does not need four different readers.

The raw files already arrive sorted by station then timestamp, so we skip the
sort. That saves a 31-million-row shuffle per file; section 3 checks the
assumption rather than trusting it.

Set `REBUILD = True` to force every file to be rewritten. Left at `False`, any
file that already has a provenance record in `data/interim/_manifest/` is kept
and its record is reported, so re-running this notebook is cheap. Delete
`data/interim/` to start completely fresh.

In [3]:
REBUILD = False

con = connect()
report = []
t_all = time.time()

for ds in DATASETS:
    for year in CFG["data"]["years"]:
        t0 = time.time()
        r = ingest_year_to_parquet(ds, year, con=con, overwrite=REBUILD)
        src_bytes = raw_file(ds, year).stat().st_size
        r["seconds"] = round(time.time() - t0, 1)
        r["mb"] = round(r["bytes"] / 1e6, 1)
        r["src_gib"] = round(src_bytes / 1024**3, 2)
        r["shrink_x"] = round(src_bytes / r["bytes"], 1)
        report.append(r)
        flagged = sum(r.get("range_flagged", {}).values())
        state = "cached " if r["skipped"] else "BUILT  "
        print(f"  {state} {ds:<6} {year}  {r['rows']:>12,} rows  ->  {r['mb']:>6.1f} MB  "
              f"({r['shrink_x']:>4.0f}x smaller)  {r['seconds']:>5.1f}s"
              + (f"   [{flagged} values failed range checks]" if flagged else ""),
              flush=True)

n_built = sum(1 for r in report if not r["skipped"])
print()
print(f"{n_built} file(s) built, {len(report) - n_built} already present "
      f"({time.time() - t_all:.0f} seconds)")

  cached  flood  2019    10,406,880 rows  ->    28.2 MB  (  42x smaller)    0.0s


  cached  flood  2020    10,435,392 rows  ->    28.3 MB  (  42x smaller)    0.0s


  cached  flood  2021    10,593,504 rows  ->    28.6 MB  (  42x smaller)    0.0s


  cached  flood  2022    10,722,240 rows  ->    29.0 MB  (  42x smaller)    0.0s


  cached  flood  2023    11,247,840 rows  ->    30.4 MB  (  42x smaller)    0.0s


...

  cached  water  2022    27,541,440 rows  ->    86.2 MB  (  47x smaller)    0.0s


  cached  water  2023    27,856,800 rows  ->    87.3 MB  (  47x smaller)    0.0s


  cached  water  2024    31,306,176 rows  ->    97.4 MB  (  48x smaller)    0.0s


  cached  water  2025    31,536,000 rows  ->    97.9 MB  (  48x smaller)    0.0s



0 file(s) built, 28 already present (0 seconds)


In [4]:
ing = pd.DataFrame(report)
ing["range_flagged_total"] = ing["range_flagged"].apply(lambda d: sum(d.values()))
ing_out = ing.drop(columns=["range_flagged", "skipped"])
ing_out["range_flagged"] = ing["range_flagged"].apply(json.dumps)
ing_out.to_csv(REPORTS / "ingest_report.csv", index=False)

src_gib = ing.src_gib.sum()
out_gib = ing.bytes.sum() / 1024**3
print(f"raw CSV   : {src_gib:>8.1f} GiB")
print(f"Parquet   : {out_gib:>8.2f} GiB")
print(f"shrinkage : {src_gib / out_gib:>8.0f}x")
print()
display(ing.pivot(index="year", columns="dataset", values="mb"))

raw CSV   :     51.1 GiB
Parquet   :     1.12 GiB
shrinkage :       46x



dataset,flood,flow,rain,water
year,,,,
2019,28.2,12.7,39.5,83.8
2020,28.3,13.9,40.5,84.4
2021,28.6,13.5,40.4,84.0
2022,29.0,14.0,41.4,86.2
2023,30.4,14.0,40.1,87.3
2024,30.5,13.9,40.8,97.4
2025,30.4,13.2,40.8,97.9


> Around 50x smaller, and that is not compression magic — it is mostly the
> removal of enormous redundancy. Every row of the CSV repeats the station code
> and its full Thai name as text; Parquet stores each one once in a dictionary
> and references it. Timestamps stop being 23-character strings and become 8-byte
> integers.

## 3. Verify: did anything change that should not have?

In [5]:
# 3a. Row counts must match the raw files exactly. Nothing may be lost or added.
inv = pd.read_csv(REPORTS / "inventory_files.csv")
check = inv[["dataset", "year", "rows"]].merge(
    ing[["dataset", "year", "rows"]], on=["dataset", "year"],
    suffixes=("_raw", "_parquet"))
check["match"] = check.rows_raw == check.rows_parquet
print(f"row counts match on {int(check.match.sum())} of {len(check)} files")
assert check.match.all(), check[~check.match]
print("OK: no row was lost or duplicated during ingestion.")

row counts match on 28 of 28 files
OK: no row was lost or duplicated during ingestion.


In [6]:
# 3b. The files really are sorted by station then time, as ingestion assumed.
orders = [verify_ordering(ds, y, con=con)
          for ds in DATASETS for y in CFG["data"]["years"]]
od = pd.DataFrame(orders)
print(f"ordered correctly: {int(od.ordered.sum())} of {len(od)} files")
assert od.ordered.all(), od[~od.ordered]
print("OK: the skip-the-sort assumption holds.")

ordered correctly: 28 of 28 files
OK: the skip-the-sort assumption holds.


In [7]:
# 3c. What did the range checks actually remove?
flags = []
for r in report:
    for col, n in r.get("range_flagged", {}).items():
        if n:
            flags.append({"dataset": r["dataset"], "year": r["year"],
                          "column": col, "values_nulled": n})
if flags:
    fl = pd.DataFrame(flags)
    fl.to_csv(REPORTS / "ingest_range_flags.csv", index=False)
    print(f"{fl.values_nulled.sum():,} values nulled by range checks "
          f"({100 * fl.values_nulled.sum() / ing['rows'].sum():.6f}% of all rows)")
    display(fl)
else:
    print("No values failed the range checks.")

3,043 values nulled by range checks (0.000774% of all rows)


,dataset,year,column,values_nulled
0,flow,2019,mean_velocity,1256
1,flow,2020,mean_velocity,41
2,flow,2021,mean_velocity,18
3,flow,2022,mean_velocity,1596
4,flow,2023,mean_velocity,96
5,rain,2023,rf3hr,2
6,rain,2023,rf6hr,3
7,rain,2023,rf12hr,27
8,rain,2023,rf24hr,3
9,rain,2024,rf15min,1


### Chasing down what was removed

A range check that fires is a claim about the data, so it is worth looking at the
actual rows rather than trusting the count.

In [8]:
from bkkflood.rawio import read_raw_sql

print("In the RAW 2023 rain file — the readings a human would call impossible:")
display(con.execute(
    "SELECT rain_code AS station_code, site_timestamp AS ts, rf5min, rf1hr, rf24hr "
    "FROM " + read_raw_sql("rain", 2023) + " "
    "WHERE rf24hr > 400 ORDER BY rf24hr DESC LIMIT 5"
).fetchdf())

print()
print("The same station in the CLEANED Parquet. The impossible value is gone,")
print("the plausible readings around it are untouched, and the row still exists:")
display(con.execute(
    "SELECT station_code, count(*) AS rows, count(rf24hr) AS rf24hr_present, "
    "       max(rf24hr) AS max_rf24hr, max(rf1hr) AS max_rf1hr, max(rf5min) AS max_rf5min "
    "FROM " + interim_sql("rain", [2023]) + " "
    "WHERE station_code = 'RF.BKY.02' GROUP BY 1"
).fetchdf())

In the RAW 2023 rain file — the readings a human would call impossible:


,station_code,ts,rf5min,rf1hr,rf24hr
0,RF.BKY.02,2023-03-29 08:35:00,0.0,60.5,762.0
1,RF.BKY.02,2023-03-29 08:30:00,0.0,66.5,762.0
2,RF.BKY.02,2023-03-29 05:40:00,0.0,28.5,418.0



The same station in the CLEANED Parquet. The impossible value is gone,
the plausible readings around it are untouched, and the row still exists:


,station_code,rows,rf24hr_present,max_rf24hr,max_rf1hr,max_rf5min
0,RF.BKY.02,105120,105117,393.5,73.0,13.0


> Note what survived: the row is still there and `rf1hr` is still 73 mm. Only the
> single impossible field became missing. That is the difference between removing
> a *value* and removing a *reading* — and it is why the check runs per column,
> not per row.
>
> **Worth asking BMA:** the maximum `rf5min` anywhere in seven years is exactly
> 30.0 mm, across many different gauges. A shared exact ceiling across independent
> instruments usually means a device or field limit rather than a physical
> maximum. If it is a cap, extreme 5-minute intensities in this archive are
> censored, and that matters for the feature that predicts flash flooding.

## 4. What the cleaned data looks like

In [9]:
for ds in DATASETS:
    print(f"--- {ds} ---")
    display(con.execute(f"SELECT * FROM {interim_sql(ds, [2022])} LIMIT 3").fetchdf())

--- flood ---


,station_code,station_name,ts,flood
0,FL.BBN.01,ถ.เอกชัย ช่วงห้างบิ๊กซีบางบอน,2022-01-01 00:00:00,0.0
1,FL.BBN.01,ถ.เอกชัย ช่วงห้างบิ๊กซีบางบอน,2022-01-01 00:05:00,0.0
2,FL.BBN.01,ถ.เอกชัย ช่วงห้างบิ๊กซีบางบอน,2022-01-01 00:10:00,0.0


--- flow ---


,station_code,station_name,ts,flow,wl,area,mean_velocity
0,FW.BBU.01,จุดวัดคลองบางบัว,2022-01-01 00:00:00,0.0,0.04,35.36,0.0
1,FW.BBU.01,จุดวัดคลองบางบัว,2022-01-01 00:05:00,0.0,0.04,35.30,0.0
2,FW.BBU.01,จุดวัดคลองบางบัว,2022-01-01 00:10:00,0.0,0.04,35.30,0.0


--- rain ---


,station_code,station_name,ts,rf5min,rf15min,rf30min,rf1hr,rf3hr,rf6hr,rf12hr,rf24hr
0,RF.BBN.01,สำนักงานเขตบางบอน,2022-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,RF.BBN.01,สำนักงานเขตบางบอน,2022-01-01 00:05:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,RF.BBN.01,สำนักงานเขตบางบอน,2022-01-01 00:10:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


--- water ---


,station_code,station_name,ts,wl_in,wl_out01,wl_out02
0,WL.BBN.01,จุดวัดคลองบางบอน ตอนถนนบางขุนเทียน,2022-01-01 00:00:00,0.35,NaN,NaN
1,WL.BBN.01,จุดวัดคลองบางบอน ตอนถนนบางขุนเทียน,2022-01-01 00:05:00,0.35,NaN,NaN
2,WL.BBN.01,จุดวัดคลองบางบอน ตอนถนนบางขุนเทียน,2022-01-01 00:10:00,0.34,NaN,NaN


In [10]:
# Every dataset now shares the same first three columns, which is the point.
schemas = []
for ds in DATASETS:
    cols = con.execute(f"SELECT * FROM {interim_sql(ds, [2019])} LIMIT 0").fetchdf().columns
    schemas.append({"dataset": ds, "columns": ", ".join(cols)})
pd.DataFrame(schemas)

,dataset,columns
0,flood,"station_code, station_name, ts, flood"
1,flow,"station_code, station_name, ts, flow, wl, area..."
2,rain,"station_code, station_name, ts, rf5min, rf15mi..."
3,water,"station_code, station_name, ts, wl_in, wl_out0..."


## 5. Summary

In [11]:
print("=" * 74)
print("INGESTION COMPLETE")
print("=" * 74)
print(f"  files present      : {len(ing)}  ({n_built} rebuilt this run)")
print(f"  rows              : {ing['rows'].sum():,}")
print(f"  raw CSV            : {ing.src_gib.sum():.1f} GiB")
print(f"  cleaned Parquet    : {ing.bytes.sum() / 1024**3:.2f} GiB "
      f"({ing.src_gib.sum() / (ing.bytes.sum() / 1024**3):.0f}x smaller)")
print(f"  values nulled      : {ing.range_flagged_total.sum():,} (failed range checks)")
print(f"  rows lost          : 0  (verified against notebook 00)")
print(f"  ordering verified  : {int(od.ordered.sum())}/{len(od)} files")
print("=" * 74)
print()
print("Next: notebook 02 builds the quality scorecard and detects flood events.")
con.close()

INGESTION COMPLETE
  files present      : 28  (0 rebuilt this run)
  rows              : 392,909,188
  raw CSV            : 51.1 GiB
  cleaned Parquet    : 1.12 GiB (46x smaller)
  values nulled      : 3,043 (failed range checks)
  rows lost          : 0  (verified against notebook 00)
  ordering verified  : 28/28 files

Next: notebook 02 builds the quality scorecard and detects flood events.
